# 01 — Capa Bronze: datos originales e integridad

## Objetivo

Este notebook sirve como guion técnico y visual para la exposición. Todas las
comprobaciones usan los artefactos completos del proyecto; las vistas de pocas filas
son únicamente para mostrar el esquema, no son el conjunto usado por el pipeline.

**QUÉ EXPLICAR:** primero diga qué problema resuelve esta etapa, después muestre la
evidencia y termine conectándola con la siguiente capa.

## Preparación

Ejecútelo desde el contenedor `spark` con la raíz `/workspace`. La celda también
encuentra el repositorio cuando el notebook se abre desde la carpeta local.

In [ ]:
from pathlib import Path
import json

# Encontrar la raíz permite usar el mismo notebook en Docker o en el repositorio local.
candidates = [Path.cwd(), Path.cwd().parent, Path('/workspace')]
ROOT = next(
    path for path in candidates
    if (path / 'config' / 'pipeline.yaml').exists()
)
DATA = ROOT / 'data'
EXPORTS = ROOT / 'exports'
print(f'Raíz del proyecto: {ROOT}')

## Pasos

### 1. Inventario y trazabilidad

Bronze guarda cada Parquet tal como lo publica TLC, más su sidecar de metadatos, SHA-256
y catálogo. Es una capa inmutable y reejecutable.

**QUÉ EXPLICAR:** Bronze responde “qué recibí, de dónde, cuándo y con qué huella”.

In [ ]:
from collections import Counter
import re

bronze = DATA / 'bronze'
files = list(bronze.rglob('*_tripdata_*.parquet'))
years = Counter(int(re.search(r'_(20\d{2})-', p.name).group(1)) for p in files)
sidecars = list(bronze.rglob('*.metadata.json'))
print('Archivos Parquet originales:', len(files))
print('Archivos por año:', dict(sorted(years.items())))
print('Sidecars:', len(sidecars))
print('Tamaño Bronze (GB):', round(sum(p.stat().st_size for p in files) / 1e9, 2))

### 2. Ejemplo de sidecar sin modificar el dato

In [ ]:
# Mostrar un sidecar es más claro que abrir un Parquet binario durante la defensa.
example = json.loads(sidecars[0].read_text(encoding='utf-8'))
print(json.dumps(example, indent=2, ensure_ascii=False)[:2500])

## Comprobaciones

In [ ]:
report = json.loads((EXPORTS / 'verification_report.json').read_text(encoding='utf-8'))
bronze_check = next(c for c in report['checks'] if c['name'].startswith('bronze_'))
assert bronze_check['passed'] and bronze_check['details']['historical_files'] == 144
assert bronze_check['details']['checksums_recomputed'] is True
print(bronze_check['details'])

## Siguiente paso

Bronze no corrige datos. La capa Silver aplica el esquema canónico, calidad, zonas y
cuarentena manteniendo reconciliación fila a fila.